# llama-3-8b-instruct few-shot in-context ner benchmark
no training. 4-bit nf4 on colab t4. ~25-30 fixed few-shot examples (picked from train, at least 2 examples per tag, all 17 tags guaranteed), greedy decoding, seqeval scoring.

before running:
1. request access to `meta-llama/Meta-Llama-3-8B-Instruct` on hugging face (approval can take a few min)
2. colab -> secrets (key icon) -> add `HF_TOKEN`, toggle notebook access on
3. runtime -> change runtime type -> T4 gpu

In [1]:
!pip -q install -U transformers accelerate bitsandbytes datasets seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [10]:
# ---------------- config (edit these) ----------------
DATASET_ID  = "Souvikbasur/17entity-news-ner-dataset"
TOKENS_COL  = "tokens"
TAGS_COL    = "ner_tags"
TRAIN_SPLIT = "train"
TEST_SPLIT  = "test"
MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"
K_PER_TAG   = 2         # min examples per tag in the prompt
MAX_SHOTS   = 30        # hard cap on total examples
N_EVAL      = 1000      # None = full test split
BATCH_SIZE  = 8         # drop to 4 if oom
MAX_NEW     = 200
SEED        = 42
OUT_JSONL   = "/content/fewshot_raw_preds.jsonl"   # resume-safe if colab disconnects

import os
from getpass import getpass
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("paste your hf token (read access): ")

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
print("logged in as:", whoami()["name"])

paste your hf token (read access): ··········
logged in as: Souvikbasur


In [11]:
import re, json
from datasets import load_dataset
from collections import Counter

raw = load_dataset(DATASET_ID, token=HF_TOKEN)["train"]
ex0 = raw[0]
print(json.dumps({"data": ex0["data"], "annotations": ex0["annotations"]}, indent=1, default=str)[:1200])

TOK_RE = re.compile(r"\w+|[^\w\s]")      # words + punctuation as separate tokens

def get_text(d):
    for k in ("text", "sentence", "Sentence"):
        if isinstance(d.get(k), str): return d[k]
    for v in d.values():
        if isinstance(v, str): return v
    return ""

def convert(ex):
    text = get_text(ex["data"])
    toks = [(m.group(), m.start(), m.end()) for m in TOK_RE.finditer(text)]
    tags = ["O"] * len(toks)
    anns = [a for a in (ex["annotations"] or []) if not a.get("was_cancelled")]
    if anns:
        res = [r for r in (anns[0].get("result") or [])
               if r.get("type") == "labels" and r["value"].get("labels")]
        res.sort(key=lambda r: r["value"]["start"] - r["value"]["end"])   # longest span first
        for r in res:
            v = r["value"]; lab = v["labels"][0]
            idxs = [k for k, (_, s, e) in enumerate(toks) if s < v["end"] and e > v["start"]]
            if not idxs or any(tags[k] != "O" for k in idxs): continue
            tags[idxs[0]] = "B-" + lab
            for k in idxs[1:]: tags[k] = "I-" + lab
    return [t for t, _, _ in toks], tags

# 80:10:10, seed 42 -> use the 10% test
sp  = raw.train_test_split(test_size=0.2, seed=42)
sp2 = sp["test"].train_test_split(test_size=0.5, seed=42)
train = [x for x in (convert(e) for e in sp["train"]) if x[0]]
test  = [x for x in (convert(e) for e in sp2["test"])  if x[0]]

LABELS = sorted({t[2:] for _, tags in train for t in tags if t != "O"})
print(len(LABELS), "entity types:", LABELS)
print("train", len(train), "test", len(test))
print(Counter(t[2:] for _, tags in train for t in tags if t.startswith("B-")).most_common())
print(train[0])

{
 "data": {
  "text": "In the 2024 presidential election , you could vote for Kamala Harris on the Democratic line or on the Working Families Party line ."
 },
 "annotations": [
  {
   "id": 1,
   "completed_by": 1,
   "result": [
    {
     "value": {
      "start": 7,
      "end": 11,
      "text": "2024",
      "labels": [
       "DATE"
      ]
     },
     "id": "z8OrxLCZCC",
     "from_name": "label",
     "to_name": "text",
     "type": "labels",
     "origin": "manual"
    },
    {
     "value": {
      "start": 55,
      "end": 68,
      "text": "Kamala Harris",
      "labels": [
       "PER"
      ]
     },
     "id": "sCd9LABXHN",
     "from_name": "label",
     "to_name": "text",
     "type": "labels",
     "origin": "manual"
    },
    {
     "value": {
      "start": 76,
      "end": 86,
      "text": "Democratic",
      "labels": [
       "NORP"
      ]
     },
     "id": "IA6QOAYluY",
     "from_name": "label",
     "to_name": "text",
     "type": "labels",
     "origin

In [12]:
def spans_of(tokens, tags):
    out, i = [], 0
    while i < len(tags):
        if tags[i].startswith("B-"):
            lab, j = tags[i][2:], i + 1
            while j < len(tags) and tags[j] == "I-" + lab:
                j += 1
            out.append((" ".join(tokens[i:j]), lab))
            i = j
        else:
            i += 1
    return out

# few-shot pick: every tag gets >= K_PER_TAG example sentences (rarest tags handled first)
import random
from collections import Counter

label_freq = Counter(l for t, g in train for _, l in spans_of(t, g))
def labs(x): return {l for _, l in spans_of(*x)}

pool = [(t, g) for t, g in train if 6 <= len(t) <= 40 and len(spans_of(t, g)) >= 1]
random.Random(SEED).shuffle(pool)
have, shots, skipped = Counter(), [], set()

def still_missing():
    return [l for l in LABELS if have[l] < K_PER_TAG and l not in skipped]

while still_missing() and len(shots) < MAX_SHOTS:
    target = min(still_missing(), key=lambda l: label_freq[l])      # rarest first
    cand = [x for x in pool if target in labs(x)]
    if not cand:                                                     # fallback: any length
        cand = [x for x in train if target in labs(x) and x not in shots]
    if not cand:
        skipped.add(target); continue
    best = max(cand, key=lambda x: (sum(1 for l in labs(x) if have[l] < K_PER_TAG), -len(x[0])))
    shots.append(best)
    if best in pool: pool.remove(best)
    for l in labs(best): have[l] += 1

random.Random(SEED).shuffle(shots)   # don't cluster tags by order

print("total shots:", len(shots))
print(f"{'tag':6} {'n_shots':>7}  example")
for l in LABELS:
    ex = next(((t, x) for t, g in shots for x, ll in spans_of(t, g) if ll == l), None)
    print(f"{l:6} {have[l]:>7}  {ex[1] if ex else '-- MISSING --'}")
missing = [l for l in LABELS if have[l] == 0]
assert not missing, f"tags with no example in prompt: {missing}"
print("all", len(LABELS), "tags covered, min examples per tag:", min(have[l] for l in LABELS))

total shots: 10
tag    n_shots  example
CARD         2  four
DATE         5  yesterday
EVT          2  the World Series
FAC          2  Dover Air Force Base
LANG         2  Chinese
LAW          2  IRS Notice 89 - 136
LOC          4  Delaware
MONEY        2  $ 46 . 8 million
NORP         2  Chinese
ORD          2  third
ORG          5  Toyota
PCT          2  only 7 %
PER          2  Gregory Lee
PROD         2  Corollas
QTY          2  34 , 500 metric tons
TIME         2  6 a . m .
WOA          2  Batibot
all 17 tags covered, min examples per tag: 2


In [13]:
DESC = {
 "PER":"person", "ORG":"organization/company/agency/party", "LOC":"country, city, region, geographic place",
 "NORP":"nationality, religious or political group", "DATE":"date or period", "TIME":"time of day",
 "MONEY":"monetary value", "PCT":"percentage", "CARD":"cardinal number", "ORD":"ordinal number",
 "QTY":"measurement/quantity", "EVT":"named event", "LAW":"law, treaty, bill", "FAC":"building, airport, bridge, facility",
 "PROD":"product, vehicle, weapon", "LANG":"language", "WOA":"work of art, book, film, title",
}
label_block = "\n".join(f"{l}: {DESC.get(l, l)}" for l in LABELS)

SYSTEM = (
 "You are an expert named entity recognition system for news text.\n"
 "Extract every named entity from the sentence and label it with exactly one of these types:\n"
 + label_block +
 "\n\nRules: copy each entity exactly as it appears in the sentence (tokens separated by single spaces). "
 "List every occurrence separately, in order of appearance. "
 'Output only a JSON list like [{"text": "...", "type": "..."}]. If there are no entities, output [].'
)

def user_msg(tokens): return "Sentence: " + " ".join(tokens)
def asst_msg(tokens, tags):
    return json.dumps([{"text": t, "type": l} for t, l in spans_of(tokens, tags)], ensure_ascii=False)

def build_messages(tokens):
    m = [{"role": "system", "content": SYSTEM}]
    for t, g in shots:
        m += [{"role": "user", "content": user_msg(t)}, {"role": "assistant", "content": asst_msg(t, g)}]
    m.append({"role": "user", "content": user_msg(tokens)})
    return m

print(SYSTEM[:600], "...")

You are an expert named entity recognition system for news text.
Extract every named entity from the sentence and label it with exactly one of these types:
CARD: cardinal number
DATE: date or period
EVT: named event
FAC: building, airport, bridge, facility
LANG: language
LAW: law, treaty, bill
LOC: country, city, region, geographic place
MONEY: monetary value
NORP: nationality, religious or political group
ORD: ordinal number
ORG: organization/company/agency/party
PCT: percentage
PER: person
PROD: product, vehicle, weapon
QTY: measurement/quantity
TIME: time of day
WOA: work of art, book, film ...


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tok.padding_side = "left"
tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto",
                                             attn_implementation="sdpa", token=HF_TOKEN)
model.eval()
STOP = [tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")]
print("gpu mem GB:", round(torch.cuda.memory_allocated() / 1e9, 2))
print("prompt tokens per sentence ~", len(tok.apply_chat_template(build_messages(test[0][0]), add_generation_prompt=True)))

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

gpu mem GB: 5.71
prompt tokens per sentence ~ 2


In [15]:
# pick eval subset (fixed seed), sort by length for efficient batching
idx = list(range(len(test)))
random.Random(SEED).shuffle(idx)
if N_EVAL: idx = idx[:N_EVAL]

done = {}
if os.path.exists(OUT_JSONL):
    for line in open(OUT_JSONL):
        r = json.loads(line); done[r["i"]] = r["raw"]
print("already done:", len(done))

todo = sorted([i for i in idx if i not in done], key=lambda i: len(test[i][0]))
t0 = time.time()
with open(OUT_JSONL, "a") as f:
    for b in range(0, len(todo), BATCH_SIZE):
        chunk = todo[b:b + BATCH_SIZE]
        prompts = [tok.apply_chat_template(build_messages(test[i][0]), add_generation_prompt=True, tokenize=False)
                   for i in chunk]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 eos_token_id=STOP, pad_token_id=tok.eos_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, raw in zip(chunk, texts):
            done[i] = raw; f.write(json.dumps({"i": i, "raw": raw}) + "\n")
        f.flush()
        n = b + len(chunk)
        if (b // BATCH_SIZE) % 10 == 0:
            el = time.time() - t0
            print(f"{n}/{len(todo)} | {el/60:.1f} min | eta {(el/n)*(len(todo)-n)/60:.1f} min")
inference_min = (time.time() - t0) / 60
print("inference time (this run):", round(inference_min, 1), "min")

already done: 0


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


8/1000 | 0.3 min | eta 38.4 min


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

88/1000 | 4.6 min | eta 47.4 min


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

... (19 outputs left out)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

968/1000 | 84.5 min | eta 2.8 min


[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


inference time (this run): 88.9 min


In [16]:
import re, json, pandas as pd
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.metrics.sequence_labeling import get_entities
from seqeval.scheme import IOB2

def parse(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    if not m: return None
    try:
        v = json.loads(m.group(0))
        return [p for p in v if isinstance(p, dict)]
    except Exception:
        return None

def align(tokens, preds):
    tags = ["O"] * len(tokens)
    for p in preds:
        t, l = p.get("text"), p.get("type")
        if not isinstance(t, str) or l not in LABELS: continue
        pt = t.split(); n = len(pt)
        if n == 0: continue
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == pt and all(x == "O" for x in tags[i:i+n]):
                tags[i] = "B-" + l
                for j in range(i + 1, i + n): tags[j] = "I-" + l
                break
    return tags

y_true, y_pred, bad = [], [], 0
for i in idx:
    toks, gold_tags = test[i]
    p = parse(done[i])
    if p is None: bad += 1; p = []
    y_true.append(gold_tags); y_pred.append(align(toks, p))
print(f"evaluated {len(idx)} sentences | unparseable outputs: {bad}")

# ---- entity-level exact match (same as seqeval strict) ----
gold, pred = set(), set()
for k, (g, p) in enumerate(zip(y_true, y_pred)):
    for lab, s, e in get_entities(g): gold.add((k, lab, s, e))
    for lab, s, e in get_entities(p): pred.add((k, lab, s, e))

def prf(tp, fp, fn):
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return P, R, F

rows = []
for l in LABELS:
    g = {x for x in gold if x[1] == l}; p = {x for x in pred if x[1] == l}
    tp = len(g & p)
    P, R, F = prf(tp, len(p) - tp, len(g) - tp)
    rows.append(dict(tag=l, precision=P, recall=R, f1=F, support=len(g), predicted=len(p), tp=tp))
df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)

has = df[df.support > 0]
macro = has[["precision", "recall", "f1"]].mean()
weighted = has[["precision", "recall", "f1"]].mul(has.support, axis=0).sum() / has.support.sum()
tp = len(gold & pred)
micro = prf(tp, len(pred) - tp, len(gold) - tp)

print("\n=== per-tag (entity-level, exact match) ===")
print(df.round(4).to_string(index=False))
print("\n=== overall ===")
print(f"macro    P {macro.precision:.4f} | R {macro.recall:.4f} | F1 {macro.f1:.4f}   <- main number (every tag equal weight)")
print(f"weighted P {weighted.precision:.4f} | R {weighted.recall:.4f} | F1 {weighted.f1:.4f}")
print(f"micro    P {micro[0]:.4f} | R {micro[1]:.4f} | F1 {micro[2]:.4f}")
print("tags with zero support in this eval subset (excluded from macro):",
      df[df.support == 0].tag.tolist())

# ---- seqeval cross-check, strict IOB2 ----
print("\n=== seqeval strict report (cross-check) ===")
print(classification_report(y_true, y_pred, mode="strict", scheme=IOB2, digits=4))

df.to_csv("/content/llama3_8b_fewshot_per_tag.csv", index=False)

evaluated 1000 sentences | unparseable outputs: 2

=== per-tag (entity-level, exact match) ===
  tag  precision  recall     f1  support  predicted  tp
 LANG     0.8333  0.9091 0.8696       11         12  10
  LOC     0.7996  0.7981 0.7989      530        529 423
  PER     0.7927  0.7019 0.7446      463        410 325
  PCT     0.7581  0.6812 0.7176       69         62  47
  WOA     0.8182  0.5000 0.6207       72         44  36
  ORG     0.5845  0.5961 0.5902      557        568 332
 DATE     0.7202  0.4373 0.5442      359        218 157
 NORP     0.5000  0.5349 0.5169      215        230 115
MONEY     0.4338  0.5784 0.4958      102        136  59
 TIME     0.4186  0.5625 0.4800       32         43  18
  EVT     0.3699  0.5400 0.4390       50         73  27
 CARD     0.5921  0.2663 0.3673      169         76  45
  LAW     0.2745  0.4118 0.3294       34         51  14
  FAC     0.2778  0.3448 0.3077       29         36  10
  QTY     0.3115  0.2923 0.3016       65         61  19
 PROD    

In [17]:
# save results for the paper table
import pandas as pd
rep = classification_report(y_true, y_pred, digits=4)
open("/content/llama3_8b_fewshot_report.txt", "w").write(
    f"model={MODEL_ID}\nshots={len(shots)}\nn_eval={len(idx)}\nunparseable={bad}\n"
    f"inference_min={inference_min:.1f}\nmicro_f1={f1_score(y_true, y_pred):.4f}\n\n{rep}")
pd.DataFrame({"i": idx, "raw": [done[i] for i in idx]}).to_csv("/content/llama3_8b_fewshot_raw.csv", index=False)
print("saved to /content: llama3_8b_fewshot_report.txt, llama3_8b_fewshot_raw.csv")

saved to /content: llama3_8b_fewshot_report.txt, llama3_8b_fewshot_raw.csv
